# Sequential Fine-Tuning

A practical reference for **sequential fine-tuning** — adapting a model through a *chain* of fine-tuning stages on different datasets/tasks, one after another, where each stage starts from the previous stage's weights. The canonical example is the modern LLM recipe itself: **base pretrain → continued (domain) pretraining → supervised/instruction fine-tuning → preference optimization (DPO/RLHF)**. Covers the core idea, the stage pipeline, the central enemy (**catastrophic forgetting**), the mitigations that tame it (rehearsal, regularization, parameter isolation), and how it compares to joint multi-task training and model merging.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**Sequential fine-tuning** trains a model on a *sequence* of stages — dataset A, then dataset B, then C — instead of mixing everything into one pool and training jointly. Each stage resumes from the checkpoint the previous stage produced, so knowledge accumulates (and, if you're not careful, erodes) along the chain. It is the practical face of **transfer learning** and **continual learning**, and it is how essentially every production LLM is actually built.

The reason this is a *topic* and not just "train twice" is the **stability–plasticity dilemma**: each new stage has to stay *plastic* enough to absorb the new data, yet *stable* enough not to overwrite what it already knew. Push too hard on the new task and you get **catastrophic forgetting** — the model aces stage B while collapsing on stage A. Most of this notebook is about ordering the stages well and keeping the earlier knowledge alive.

### What is it?

A pipeline of fine-tuning runs `θ₀ → θ₁ → θ₂ → …`, where stage *i* initializes from `θ_{i-1}` and trains on dataset *Dᵢ* with its own hyperparameters (data mix, epochs, learning rate). The stages typically move **general → specific**: broad domain text first, then task/instruction data, then narrow preference or style alignment. Each stage is an ordinary fine-tuning job; the engineering is in *what order*, *what learning rate*, and *what to carry forward* so earlier stages survive.

### Why use it?

Key benefits of using sequential fine-tuning:

- **It's the natural shape of the data.** Different supervision arrives at different times and in different forms (raw domain text, labeled tasks, human preferences). Sequential stages let each be used in the form and objective that fits it — you can't express "next-token on a corpus" and "preference ranking" as one joint loss.
- **Cheap incremental updates.** When new data (a new domain, a new language, fresh preferences) shows up, you add a stage on top of the existing checkpoint instead of re-running the whole pipeline from scratch.
- **Curriculum benefits.** Learning easy/general material before hard/specific material often reaches a better optimum than dumping it all in at once — the early stage shapes good representations the later stage refines.
- **Modularity & reuse.** A domain-adapted checkpoint (e.g. a legal or medical base) can be the shared starting point for many downstream task-specific stages.

### When to use it?

Sequential fine-tuning is particularly useful when:

- Your supervision comes in **distinct phases or objectives** (pretraining-style text, then instructions, then preferences).
- You need to **adapt to a domain first, then specialize** to a task within it (general LM → BioBERT-style domain model → clinical NER).
- New data arrives **over time** and re-training jointly each time is too expensive (continual / lifelong learning).
- You're building on a checkpoint someone else trained and only have access to **add stages**, not redo the earlier ones.

If you have *all* the data up front, in a *single* objective, and can afford to train on it together, **joint multi-task fine-tuning usually forgets less** — see the comparison section.

## Key Features

### Core concepts and the levers you actually tune

| Concept | Description | Why it matters |
|---------|-------------|----------------|
| **Stage chain** | `θ₀ → θ₁ → θ₂ …`; each stage resumes from the prior checkpoint | The defining structure; lineage must be tracked so you can reproduce and roll back |
| **Catastrophic forgetting** | New-stage training overwrites weights critical to earlier stages | The central failure mode — the model "wins" the last task and loses the rest |
| **Stability–plasticity trade-off** | Learn the new task (plasticity) vs retain the old (stability) | Every mitigation is a knob on this trade-off |
| **Stage ordering / curriculum** | The sequence of stages, usually general → specific | Order changes the final optimum; later stages overwrite earlier ones, not vice-versa |
| **Learning-rate decay across stages** | Later, narrower stages use smaller LRs and fewer epochs | A high LR late in the chain is the fastest way to erase earlier knowledge |
| **Rehearsal / replay** | Mix a fraction of earlier-stage data into later stages | The simplest, most reliable anti-forgetting tool when you still have the old data |
| **Regularization (EWC, L2-SP, LwF)** | Anchor important weights / distill old behavior, no old data needed | Retains earlier stages when the original data is gone or private |
| **Parameter isolation (per-stage LoRA/adapters)** | Freeze the base; give each stage its own small adapter | Sidesteps forgetting entirely by not overwriting shared weights |

Practical guidance: order stages **general → specific**, **decay the learning rate** at every step, and **replay a small slice of earlier data** when you have it. Reach for regularization (EWC) or per-stage **adapters** when you can't replay (the old data is unavailable or you must keep stages isolated).

## Architecture Overview

Sequential fine-tuning is a **DAG of training jobs** wired checkpoint-to-checkpoint. The output of one stage is the input of the next; each stage owns its dataset, objective, and hyperparameters:

```
   base model θ0
        │
        ▼
  ┌───────────────┐   D1: domain corpus        ckpt θ1
  │  STAGE 1      │   objective: next-token  ─────────────┐
  │  continued    │   lr 2e-5, 1 epoch                    │
  │  pretraining  │                                       ▼
  └───────────────┘                            ┌───────────────┐   D2: instructions     ckpt θ2
                                               │  STAGE 2      │   objective: SFT     ─────────────┐
              replay ◄── slice of D1 ─────────►│  instruction  │   lr 1e-5, 3 epochs              │
              (anti-forgetting)                │  fine-tuning  │                                   ▼
                                               └───────────────┘                        ┌───────────────┐  D3: preferences
                                                                                        │  STAGE 3      │  objective: DPO
                                          replay ◄── slice of D1+D2 ───────────────────►│  preference   │  lr 5e-6, 1 epoch
                                                                                        │  optimization │
                                                                                        └───────┬───────┘
                                                                                                ▼
                                                                                       aligned model θ3  →  serve
```

Notice the learning rate **decays** down the chain (2e-5 → 1e-5 → 5e-6) and a **replay buffer** of earlier data is optionally fed back into each later stage — those are the two structural defenses against forgetting.

### Components

1. **Base checkpoint (`θ₀`).** The starting weights — a pretrained foundation model, or a checkpoint from an upstream team.
2. **Stage jobs.** Each is a self-contained fine-tuning run with its own dataset `Dᵢ`, objective (causal-LM, SFT, DPO, classification…), and hyperparameters. Output: `θᵢ`.
3. **Checkpoint store / lineage.** A registry that records *which checkpoint came from which stage with which data and config* — essential for reproducibility and rollback.
4. **Replay buffer (optional).** A retained sample of earlier-stage data injected into later stages to fight forgetting.
5. **Per-stage evaluation gate.** After every stage you must eval on **all previous tasks**, not just the current one — that's how you catch forgetting before it ships.
6. **Mitigation layer (optional).** Regularizers (EWC/L2-SP/LwF) or parameter-isolation (LoRA adapters per stage) bolted onto stage jobs to bound how much they disturb prior knowledge.

## Installation

### Prerequisites

- Python 3.9+
- NumPy (the self-contained forgetting/replay demos below run on CPU, no GPU or ML framework needed)
- For real LLM stages: PyTorch + an NVIDIA GPU, plus the Hugging Face stack (`transformers`, `trl`, `peft`, `datasets`, `accelerate`)
- A **checkpoint/experiment tracker** (a model registry, or even disciplined directory naming) — sequential chains are unreproducible without lineage

### Installation Steps

**Note**: the demo cells need only NumPy. Uncomment below to install the full stack for real multi-stage LLM fine-tuning.

In [ ]:
# The NumPy demos below need nothing extra. For real sequential LLM fine-tuning:
# %pip install -U transformers trl peft accelerate datasets
# Optional experiment / checkpoint-lineage tracking:
# %pip install -U wandb
import numpy as np  # noqa: F401  (used by the demo cells below)

## Basic Usage

### Quick Start Example

The whole point — and the whole danger — of sequential fine-tuning fits in one pure-NumPy demo. We train a tiny MLP on **Task A**, then *continue* training it on **Task B**, and watch Task A's accuracy collapse. Each example carries a context bit telling the model which task it is, so the network has *capacity* to hold both — meaning the forgetting we see is a **training artifact**, exactly the catastrophic forgetting that haunts real multi-stage pipelines, not a fundamental limit.

In [ ]:
# Sequential fine-tuning distilled to pure NumPy: train one tiny MLP on Task A,
# then *continue* on Task B, and watch Task A accuracy collapse -> catastrophic forgetting.
import numpy as np

rng = np.random.default_rng(0)

# Each example carries a context bit (last feature) marking which task it is, so the
# model *can* represent both at once -- the forgetting below is a training artifact,
# not a capacity limit.
def make_task(feat, ctx, n=600):
    X = rng.normal(size=(n, 2))
    y = (X[:, feat] > 0).astype(float)               # label depends on one feature
    X = np.hstack([X, np.full((n, 1), float(ctx))])  # append the context bit
    return X, y

XA, yA = make_task(feat=0, ctx=0.0)   # Task A: label = sign(feature 0)
XB, yB = make_task(feat=1, ctx=1.0)   # Task B: label = sign(feature 1)

H = 16
def init():
    return {"W1": rng.normal(scale=0.5, size=(3, H)), "b1": np.zeros(H),
            "W2": rng.normal(scale=0.5, size=(H, 1)), "b2": np.zeros(1)}

def forward(p, X):
    h = np.tanh(X @ p["W1"] + p["b1"])
    return (h @ p["W2"] + p["b2"])[:, 0], h

def acc(p, X, y):
    logit, _ = forward(p, X)
    return float(((logit > 0) == (y > 0.5)).mean())

def grads(p, X, y):
    logit, h = forward(p, X)
    d = (1.0 / (1.0 + np.exp(-logit)) - y) / len(y)   # dL/dlogit for logistic loss
    g = {"W2": h.T @ d[:, None], "b2": np.array([d.sum()])}
    dh = (d[:, None] @ p["W2"].T) * (1 - h ** 2)
    g["W1"] = X.T @ dh
    g["b1"] = dh.sum(0)
    return g

def train(p, X, y, epochs=600, lr=0.5):
    for _ in range(epochs):
        g = grads(p, X, y)
        for k in p:
            p[k] -= lr * g[k]
    return p

p = init()
train(p, XA, yA)
print("after stage 1 (Task A):  A=%.2f  B=%.2f" % (acc(p, XA, yA), acc(p, XB, yB)))
train(p, XB, yB)
print("after stage 2 (Task B):  A=%.2f  B=%.2f   <- Task A forgotten" % (acc(p, XA, yA), acc(p, XB, yB)))

### The real-world shape

In production every stage is just an ordinary `Trainer` run that **loads the previous stage's checkpoint** as its starting model and saves a new one for the next stage to consume. The art is in the table of stages — ordering, epochs, and the **decaying learning rate** — not in any special API.

In [ ]:
# Each stage = a normal fine-tune that STARTS FROM the previous stage's checkpoint.
# Sequential fine-tuning is mostly pipeline plumbing + a decaying LR schedule.
# Gated behind try/except so the notebook runs even without the libraries installed.
try:
    from transformers import (AutoModelForCausalLM, AutoTokenizer,  # noqa: F401
                              Trainer, TrainingArguments)

    BASE = "meta-llama/Llama-3.2-1B"
    STAGES = [
        # (name,                data,            epochs,  lr)   general -> specific, LR decays
        ("continued-pretrain",  "domain_corpus",      1, 2e-5),  # absorb domain text
        ("sft-instructions",    "instruction_ds",     3, 1e-5),  # learn to follow tasks
        ("dpo-preferences",     "preference_ds",      1, 5e-6),  # align to preferences
    ]

    ckpt = BASE
    for name, data, epochs, lr in STAGES:
        print(f"stage '{name}': start={ckpt}  data={data}  epochs={epochs}  lr={lr}")
        # model = AutoModelForCausalLM.from_pretrained(ckpt)   # <- load PREVIOUS stage
        # args  = TrainingArguments(output_dir=f"/tmp/{name}", num_train_epochs=epochs,
        #                           learning_rate=lr, warmup_ratio=0.03)
        # Trainer(model=model, args=args, train_dataset=load(data)).train()
        # model.save_pretrained(f"/tmp/{name}")
        ckpt = f"/tmp/{name}"   # this stage's OUTPUT becomes the next stage's INPUT
    print("Each stage's output checkpoint feeds the next -> a sequential chain.")
except Exception as e:  # noqa: BLE001 - libraries optional in this environment
    print("transformers not installed - showing the pipeline shape only:", type(e).__name__)

## Advanced Features

### Fighting catastrophic forgetting

The basic demo loses Task A. Three families of mitigation buy it back, trading off data availability, compute, and how much they isolate the stages.

#### 1. Rehearsal / replay (the workhorse)

Keep a slice of each earlier stage's data and **mix it into later stages**. The optimizer sees old and new objectives together, so it can't drift off the old one. Even ~5–25% replay typically erases most forgetting. Variants: **experience replay** (store raw examples), **generative/self replay** (sample synthetic old-task data from a frozen copy of the model when you can't keep the originals). Cost: you need access to — or a stand-in for — the old data.

#### 2. Regularization (anchor the weights, no old data)

Add a penalty that resists moving the parameters that mattered for earlier stages:

- **EWC (Elastic Weight Consolidation):** penalize change weighted by each parameter's **Fisher information** (its importance to the old task): `L = L_new + (λ/2)·Σᵢ Fᵢ(θᵢ − θ*ᵢ)²`. Important weights are stiff; unimportant ones stay free.
- **L2-SP:** a plain L2 pull back toward the starting weights `θ*` — EWC's cheap, uniform-importance cousin.
- **Learning without Forgetting (LwF):** **knowledge distillation** from a frozen copy of the previous model — match its outputs on the new data, so old behavior is preserved without old labels.

#### 3. Parameter isolation (don't overwrite at all)

Freeze the base model and give each stage its **own** small set of parameters — a per-stage **LoRA adapter** or adapter module. Earlier stages literally cannot be overwritten because their weights are frozen; you swap or compose adapters at serve time (**AdapterFusion**, multi-LoRA). This sidesteps forgetting at the cost of some cross-stage transfer (the stages share less).

#### Curriculum & ordering

Order is a lever, not a detail. Because later stages overwrite earlier ones, put the **most general, most disposable** knowledge first and the **most important, most specific** behavior last (or replay it). Easy-to-hard curricula and competence-based ordering can reach better optima than random mixing.

In [ ]:
# Mitigation #1 in action: rehearsal/replay. When training the later stage, mix in a
# slice of the earlier stage's data. Even a small fraction sharply reduces forgetting.
base = init(); train(base, XA, yA)                  # checkpoint after stage 1
for frac in (0.0, 0.25, 1.0):
    p2 = {k: v.copy() for k, v in base.items()}     # always resume from the post-stage-1 weights
    nA = int(frac * len(yA))
    idx = rng.permutation(len(yA))[:nA]
    Xmix = np.vstack([XB, XA[idx]]) if nA else XB
    ymix = np.concatenate([yB, yA[idx]]) if nA else yB
    train(p2, Xmix, ymix)
    print("replay=%3.0f%% of Task A  ->  A=%.2f  B=%.2f" % (100 * frac, acc(p2, XA, yA), acc(p2, XB, yB)))

## Use Cases

### Real-world applications of sequential fine-tuning

#### Use Case 1: Domain adaptation, then task specialization

- **Context:** You need a clinical entity extractor. A general LM knows English but little medicine; your labeled NER set is small.
- **Implementation:** Stage 1 — **continued pretraining** on a large unlabeled medical corpus (next-token objective) to shift the representations into the domain (the BioBERT / PubMedBERT recipe). Stage 2 — **supervised fine-tuning** on the small labeled NER set. Replay a little general text in stage 2 to keep fluency.
- **Results:** Markedly higher downstream accuracy than fine-tuning the general model directly, because stage 1 supplied domain representations the tiny labeled set could never learn alone.

#### Use Case 2: The standard LLM alignment pipeline

- **Context:** Turn a raw pretrained base model into a helpful, safe chat assistant.
- **Implementation:** **Continued pretraining** (optional domain/recency) → **SFT** on instruction–response demonstrations → **preference optimization** (DPO/RLHF). Each stage starts from the last, with a decaying LR and shrinking, more-curated data.
- **Results:** This sequential recipe is how essentially every production instruction-tuned model is built; each stage adds a capability the previous objective couldn't express.

#### Use Case 3: Continual / progressive updates over time

- **Context:** A deployed model must absorb a new language, a new product vertical, or a fresh batch of preference data every quarter without a full retrain.
- **Implementation:** Add a stage on top of the live checkpoint, guarded by replay or a per-stage adapter, and gate it on an eval suite covering **all** prior capabilities before promotion.
- **Results:** Cheap incremental improvement — provided forgetting is monitored; the failure mode is silent regression on old capabilities.

## Best Practices

### Recommended practices for sequential fine-tuning

1. **Order general → specific.** Broad, abundant, disposable knowledge first; narrow, precious, behavior-defining stages last. Later stages overwrite earlier ones — spend that overwriting budget wisely.
2. **Decay the learning rate every stage.** A high LR in a late, narrow stage is the single fastest way to erase everything before it. Late stages should be small LR, few epochs.
3. **Always evaluate on *all* previous tasks after each stage.** A stage that improves the current task while silently wrecking an earlier one looks like a win on the only metric you were watching. Track per-stage retention.
4. **Replay a small slice of earlier data when you have it.** It's the cheapest, most reliable anti-forgetting tool; 5–25% is often plenty (see the demo).
5. **Reach for adapters/LoRA when stages must stay isolated** (e.g. per-customer, per-domain) or when you can't replay — frozen base weights can't be forgotten.
6. **Track checkpoint lineage like source control.** Record base id, dataset, hyperparameters, and parent checkpoint for every stage. A sequential chain you can't reproduce or roll back is a liability.
7. **Warm up the LR at the start of each stage.** Jumping straight to peak LR on a fresh data distribution shocks the weights and accelerates forgetting.

## Common Pitfalls

### What to avoid when using sequential fine-tuning

1. **Catastrophic forgetting (the cardinal sin).** Optimizing the last stage in isolation tanks every earlier capability. *Avoid by* replay, regularization (EWC/L2-SP/LwF), or per-stage adapters — and by always evaluating earlier tasks.
2. **Evaluating only the current stage.** If your eval suite covers just the latest task, regressions on prior tasks are invisible until users find them. *Avoid by* keeping a cumulative eval suite that grows with every stage.
3. **Learning rate too high late in the chain.** The most common mechanical cause of forgetting. *Avoid by* decaying the LR per stage and using few epochs late.
4. **Order sensitivity / path dependence.** A→B and B→A give different models; results are sensitive to stage order and seed. *Avoid by* deliberately choosing order (general→specific) and logging it as a first-class hyperparameter.
5. **Overfitting a small final stage.** Late stages are often tiny and curated; many epochs at any LR will memorize them and overwrite earlier breadth. *Avoid by* limiting epochs, mixing in replay, or using adapters.
6. **Lost lineage.** Nobody remembers which data/config produced the shipped checkpoint, so it can't be reproduced or rolled back. *Avoid by* registering every stage's parent, data, and config.

## Performance Optimization

### Optimizing sequential fine-tuning for production

Two axes matter: **compute** (each stage is a full training run) and **retention quality** (how little you forget). They trade off — replay and regularization cost compute/data but buy retention.

#### Measure forgetting, don't eyeball it

The standard continual-learning scorecard builds an accuracy matrix `R`, where `R[i, j]` is accuracy on task *j* after finishing stage *i*, then summarizes it:

- **ACC** — mean accuracy over all tasks at the *end* of the chain (higher is better).
- **BWT (backward transfer)** — how much earlier tasks changed after later stages; **negative BWT is forgetting**, near-zero is good.

#### Key parameters to optimize

- **Replay ratio.** The dial between throughput and retention; tune it on the ACC/BWT curve rather than guessing.
- **Per-stage LR & epochs.** Smaller/fewer for late, narrow stages; this is free retention.
- **Parameter-efficient stages (LoRA/QLoRA).** Train a small adapter per stage instead of all weights — far less compute *and* near-zero forgetting of the frozen base.
- **Skip redundant re-runs.** Cache each stage's checkpoint so adding a new final stage doesn't recompute the chain.

In [ ]:
# The standard continual-learning scorecard: accuracy matrix R, then ACC and BWT.
#   R[i, j] = accuracy on task j after finishing stage i
#   ACC     = mean accuracy over all tasks at the end        (higher = better)
#   BWT     = backward transfer; how much earlier tasks moved (negative = forgetting)
def run_pipeline(replay_frac=0.0):
    p = init(); R = []
    train(p, XA, yA)
    R.append([acc(p, XA, yA), acc(p, XB, yB)])          # row 0: after stage 1
    nA = int(replay_frac * len(yA)); idx = rng.permutation(len(yA))[:nA]
    Xmix = np.vstack([XB, XA[idx]]) if nA else XB
    ymix = np.concatenate([yB, yA[idx]]) if nA else yB
    train(p, Xmix, ymix)
    R.append([acc(p, XA, yA), acc(p, XB, yB)])          # row 1: after stage 2
    return np.array(R)

def scorecard(R):
    T = R.shape[1]
    acc_final = R[-1].mean()
    bwt = np.mean([R[-1, i] - R[i, i] for i in range(T - 1)])  # over tasks learned before the last
    return acc_final, bwt

for name, frac in (("naive  ", 0.0), ("replay ", 0.25)):
    a, b = scorecard(run_pipeline(frac))
    print("%s ACC=%.2f  BWT=%+.2f" % (name, a, b))
print("Replay lifts final ACC and pushes BWT toward 0 -- forgetting nearly eliminated.")

## Production Deployment

### Deploying sequential fine-tuning in production

Sequential fine-tuning is a **training-pipeline** concern: you run the chain of stages offline and **deploy the final checkpoint** like any other model. The discipline that makes it production-grade is treating the chain as a reproducible, gated DAG — each stage a job, each output a registered, eval-gated artifact.

- **Orchestrate stages as a DAG.** Tools like Argo Workflows, Kubeflow Pipelines, Airflow, or Metaflow model "stage 2 depends on stage 1's checkpoint" natively. Each node trains, evals on *all* prior tasks, and only promotes the checkpoint if the gate passes.
- **Register every checkpoint with lineage.** Parent checkpoint, dataset version, hyperparameters, and the eval scorecard — so any stage is reproducible and rollback is one pointer change.
- **Gate, then canary.** Block promotion on the cumulative eval suite (catch forgetting), then canary the final model before full rollout.

#### Docker Deployment

```dockerfile
# One reusable image runs any stage; the stage is chosen by CLI args at runtime.
FROM nvidia/cuda:12.4.1-runtime-ubuntu22.04
RUN apt-get update && apt-get install -y python3-pip git && rm -rf /var/lib/apt/lists/*
RUN pip3 install --no-cache-dir transformers trl peft accelerate datasets
COPY train_stage.py /app/train_stage.py
WORKDIR /app
# --init-ckpt = previous stage's output; --data/--lr/--epochs define this stage.
ENTRYPOINT ["accelerate", "launch", "train_stage.py"]
```

#### Kubernetes Deployment

```yaml
# Argo Workflow: each stage is a step whose input checkpoint is the prior step's output.
apiVersion: argoproj.io/v1alpha1
kind: Workflow
metadata:
  generateName: seqft-pipeline-
spec:
  entrypoint: chain
  templates:
    - name: chain
      steps:
        - - name: stage1
            template: train
            arguments:
              parameters:
                - {name: init, value: "/models/base"}
                - {name: data, value: "domain_corpus"}
                - {name: lr,   value: "2e-5"}
                - {name: out,  value: "/models/s1"}
        - - name: stage2
            template: train
            arguments:
              parameters:
                - {name: init, value: "/models/s1"}
                - {name: data, value: "instruction_ds"}
                - {name: lr,   value: "1e-5"}
                - {name: out,  value: "/models/s2"}
        - - name: stage3
            template: train
            arguments:
              parameters:
                - {name: init, value: "/models/s2"}
                - {name: data, value: "preference_ds"}
                - {name: lr,   value: "5e-6"}
                - {name: out,  value: "/models/final"}
    - name: train
      inputs:
        parameters: [{name: init}, {name: data}, {name: lr}, {name: out}]
      container:
        image: registry.example.com/seqft-trainer:1.0.0
        args: ["--init-ckpt", "{{inputs.parameters.init}}", "--data", "{{inputs.parameters.data}}",
               "--lr", "{{inputs.parameters.lr}}", "--output", "{{inputs.parameters.out}}"]
        resources: {limits: {nvidia.com/gpu: 8}}
```

## Monitoring and Observability

### Monitoring sequential fine-tuning in production

#### Key metrics to track

- **Per-stage retention matrix (`R`, ACC, BWT).** After every stage, eval on *all* prior tasks and record the row. Negative **BWT** is the alarm that a stage is forgetting earlier capabilities.
- **Current-stage learning curve** (loss, task metric) — confirms the new stage is actually learning, not just damaging old ones.
- **Drift from the previous checkpoint** — KL / parameter-norm change vs `θ_{i-1}`; a large jump correlates with heavy forgetting.
- **Replay composition** — the actual ratio of old:new examples per batch; silent drift here changes the stability–plasticity balance.

#### Logging best practices

- Log the **full retention matrix `R`** every stage, not just the latest task's score — forgetting is only visible across tasks.
- Record **stage lineage** with every run: parent checkpoint id, dataset version, LR/epochs/replay-ratio. Sequential results are seed- and order-sensitive, so reproducibility lives or dies here.
- Snapshot **sample generations on earlier tasks** after each stage so a human can eyeball regressions the metrics miss.
- Use appropriate levels: per-step scalars at INFO, full eval matrices and sample outputs at DEBUG, BWT-below-threshold (forgetting) at WARN, NaN/divergence at ERROR.

## Troubleshooting

### Common issues with sequential fine-tuning

#### Issue 1: Later stage tanks earlier-task accuracy

**Symptoms**: Stage *N* hits its target metric, but tasks from stages 1…N−1 regress sharply (large negative BWT).

**Cause**: Catastrophic forgetting — the stage overwrote weights critical to earlier tasks, usually from too-high LR, too many epochs, or no retention mechanism.

**Solution**: Lower the LR and epochs for the stage, add **replay** of earlier data (even 5–25%), or add an **EWC/L2-SP** penalty; if stages must stay independent, switch to **per-stage LoRA adapters**.

#### Issue 2: The later stage won't learn / underfits

**Symptoms**: New-stage loss barely moves; the model stays glued to its previous behavior.

**Cause**: Over-strong regularization (EWC `λ` too high, or replay ratio so high the new task is drowned out), or LR decayed too aggressively.

**Solution**: Reduce the regularization strength / replay fraction, raise the stage LR a bit, or add warmup — rebalance the stability–plasticity dial toward plasticity.

#### Issue 3: Results aren't reproducible between runs

**Symptoms**: The "same" pipeline yields materially different final models.

**Cause**: Sequential chains are sensitive to **stage order, seed, and data shuffling**, and lineage wasn't pinned.

**Solution**: Fix seeds, pin dataset versions and stage order, and register each stage's parent checkpoint + config. Treat order as an explicit, logged hyperparameter, not an accident of scheduling.

## Comparison with Alternatives

### How sequential fine-tuning compares to other strategies

| Dimension | Sequential fine-tuning | Joint multi-task fine-tuning | Model merging | Single-stage fine-tuning |
|-----------|------------------------|------------------------------|---------------|--------------------------|
| Data needed at once | Stages can arrive over time | All tasks up front, mixed | Independently trained models | One task's data |
| Forgetting risk | High (needs mitigation) | Low (tasks trained together) | Low (combine after the fact) | N/A (one task) |
| Different objectives per phase | Yes (LM → SFT → DPO) | Hard (one joint loss) | Yes (merge separate runs) | One objective |
| Compute | Sum of stages; incremental updates cheap | One big run | Train separately + cheap merge | One run |
| Best when | Phased supervision / continual updates | All data available, want robustness | Combining specialists post-hoc | A single, well-defined task |

### When to choose sequential fine-tuning

Choose it when:

- Supervision arrives in **distinct phases or objectives** that can't be expressed as one joint loss (the LM→SFT→preference recipe).
- You must **adapt to a domain first, then specialize**, or absorb **new data over time** without a full retrain.
- You're **building on an upstream checkpoint** and can only add stages.

Prefer **joint multi-task fine-tuning** when you have all the data up front in compatible objectives and want maximum robustness to forgetting; prefer **model merging** (weight averaging / task arithmetic) when you've already trained separate specialists and want to combine them cheaply; prefer a **single stage** when one well-defined task is all you have. In practice these compose — a sequential pipeline whose individual stages use LoRA, with merging used to fold specialists together. See the companion notebooks on Continued Pre-Training, Supervised / Instruction Fine-Tuning, Transfer Learning, and Reinforcement Learning.

## Resources

### Official Documentation

- Hugging Face Transformers (Trainer / fine-tuning): https://huggingface.co/docs/transformers/training
- Hugging Face PEFT (LoRA / adapters for per-stage isolation): https://huggingface.co/docs/peft
- Hugging Face TRL (SFT / DPO stages): https://huggingface.co/docs/trl

### Papers

- Overcoming catastrophic forgetting in neural networks (EWC) — https://arxiv.org/abs/1612.00796
- Gradient Episodic Memory for Continual Learning (defines ACC / BWT) — https://arxiv.org/abs/1706.08840
- Learning without Forgetting (LwF) — https://arxiv.org/abs/1606.09282
- Don't Stop Pretraining: Adapt Language Models to Domains and Tasks — https://arxiv.org/abs/2004.10964
- BioBERT: a pre-trained biomedical language representation model — https://arxiv.org/abs/1901.08746

### Tutorials and Guides

- Hugging Face — fine-tuning a pretrained model: https://huggingface.co/docs/transformers/training
- Avalanche (continual-learning library: replay/EWC/strategies): https://avalanche.continualai.org/
- Sebastian Ruder — Transfer Learning in NLP: https://ruder.io/transfer-learning/

### Community Resources

- Hugging Face forums — https://discuss.huggingface.co/
- ContinualAI community — https://www.continualai.org/
- Stack Overflow tag — https://stackoverflow.com/questions/tagged/transfer-learning

### Related Technologies

- Continued Pre-Training and Domain-Specific Fine-Tuning (companion notebooks — typical early stages)
- Parameter-Efficient Fine-Tuning (LoRA/QLoRA — per-stage isolation that avoids forgetting)
- Reinforcement Learning / DPO (companion notebook — a typical final stage)
- Model merging & task arithmetic (the post-hoc alternative to sequential chaining)